# Reversible Circuits Project: Phase 1

Circuit model, simulator, and the transformation rules of Feng and Li
(*A Complete Set of Transformation Rules for Reversible Circuits*, IEEE TCAD, 2026).

**How to run:** choose **Runtime > Run all** from the menu. The whole notebook takes a few seconds.

| Step | What it does |
|---|---|
| 1 | Creates `revcircuit.py`, the circuit model and simulator |
| 2 | Creates `rules.py`, Rules 1 to 5 and derived Rules 6 to 10 |
| 3 | Creates `verify_phase1.py`, the exhaustive test of every rule |
| 4 | Runs the test |
| 5 | A small example you can edit and re-run |

Colab deletes the `.py` files when the session ends. Running the notebook again recreates them.

## Step 1: Circuit model and simulator

In [ ]:
%%writefile revcircuit.py
"""Reversible circuit model and simulator (Phase 1).

Gates are mixed-polarity multiple-control Toffoli (MPMCT) gates, written
G[P, N, q] as in Feng and Li, "A Complete Set of Transformation Rules for
Reversible Circuits", IEEE TCAD, 2026:

    P  set of positive control lines
    N  set of negative control lines
    q  target line

The gate flips q when every line in P is 1 and every line in N is 0.
Special cases: X[q] = G[{}, {}, q], CNOT[p, q] = G[{p}, {}, q], and a
Toffoli gate is G[{a, b}, {}, q].

A basis state is stored as an integer whose bit i is the value of line q_i.
A circuit is a list of gates; the leftmost gate executes first.
"""
from __future__ import annotations

import random
from dataclasses import dataclass
from itertools import product
from typing import Iterable, Iterator, List, Optional, Sequence, Tuple


# ---------------------------------------------------------------------------
# Gate
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Gate:
    """One MPMCT gate G[pos, neg, target]."""

    pos: frozenset
    neg: frozenset
    target: int

    def __post_init__(self) -> None:
        object.__setattr__(self, "pos", frozenset(self.pos))
        object.__setattr__(self, "neg", frozenset(self.neg))
        if self.pos & self.neg:
            raise ValueError("a line cannot be both a positive and a negative control")
        if self.target in self.pos or self.target in self.neg:
            raise ValueError("the target line cannot also be a control")
        if min(self.lines) < 0:
            raise ValueError("line numbers must be non-negative")
        # Bit masks used by the simulator.
        object.__setattr__(self, "_pmask", sum(1 << l for l in self.pos))
        object.__setattr__(self, "_nmask", sum(1 << l for l in self.neg))
        object.__setattr__(self, "_tmask", 1 << self.target)

    @property
    def controls(self) -> frozenset:
        return self.pos | self.neg

    @property
    def lines(self) -> frozenset:
        return self.pos | self.neg | {self.target}

    def apply(self, state: int) -> int:
        """Return the state after this gate acts on `state`."""
        if (state & self._pmask) == self._pmask and not (state & self._nmask):
            return state ^ self._tmask
        return state

    def __str__(self) -> str:
        t = self.target
        if not self.neg:
            if not self.pos:
                return f"X[q{t}]"
            if len(self.pos) == 1:
                (c,) = self.pos
                return f"CNOT[q{c},q{t}]"

        def fmt(s: frozenset) -> str:
            return "{" + ",".join(f"q{l}" for l in sorted(s)) + "}"

        return f"G[{fmt(self.pos)},{fmt(self.neg)},q{t}]"


def G(pos: Iterable[int], neg: Iterable[int], target: int) -> Gate:
    """General gate G[P, N, q]."""
    return Gate(frozenset(pos), frozenset(neg), target)


def X(q: int) -> Gate:
    """NOT gate on line q."""
    return Gate(frozenset(), frozenset(), q)


def CNOT(control: int, target: int) -> Gate:
    """Controlled-NOT (Feynman gate)."""
    return Gate(frozenset({control}), frozenset(), target)


def TOF(c1: int, c2: int, target: int) -> Gate:
    """Toffoli gate with two positive controls."""
    return Gate(frozenset({c1, c2}), frozenset(), target)


def MCT(controls: Iterable[int], target: int) -> Gate:
    """Multiple-control Toffoli gate with positive controls only."""
    return Gate(frozenset(controls), frozenset(), target)


# ---------------------------------------------------------------------------
# Circuit
# ---------------------------------------------------------------------------
class Circuit:
    """A reversible circuit on n lines: an ordered list of gates."""

    def __init__(self, n: int, gates: Iterable[Gate] = ()) -> None:
        self.n = n
        self.gates: List[Gate] = list(gates)
        for g in self.gates:
            if max(g.lines) >= n:
                raise ValueError(f"gate {g} uses a line outside q0..q{n - 1}")

    # -- container behaviour ------------------------------------------------
    def __len__(self) -> int:
        return len(self.gates)

    def __getitem__(self, i: int) -> Gate:
        return self.gates[i]

    def __iter__(self) -> Iterator[Gate]:
        return iter(self.gates)

    def __eq__(self, other: object) -> bool:
        """Structural equality: same lines and the same gate list."""
        return isinstance(other, Circuit) and self.n == other.n and self.gates == other.gates

    def __add__(self, other: "Circuit") -> "Circuit":
        """Concatenation: self executes first, then other."""
        if self.n != other.n:
            raise ValueError("circuits must have the same number of lines")
        return Circuit(self.n, self.gates + other.gates)

    def __str__(self) -> str:
        return " ".join(str(g) for g in self.gates) if self.gates else "(empty)"

    def copy(self) -> "Circuit":
        return Circuit(self.n, self.gates)

    def replace(self, i: int, j: int, new_gates: Sequence[Gate]) -> "Circuit":
        """Return a new circuit with gates[i:j] replaced by new_gates."""
        return Circuit(self.n, self.gates[:i] + list(new_gates) + self.gates[j:])

    def inverse(self) -> "Circuit":
        """Every gate is its own inverse, so the inverse is the reversed list."""
        return Circuit(self.n, reversed(self.gates))

    # -- simulation ---------------------------------------------------------
    def apply(self, state: int) -> int:
        for g in self.gates:
            state = g.apply(state)
        return state

    def permutation(self) -> Tuple[int, ...]:
        """The reversible function computed, as a tuple over all 2^n states."""
        return tuple(self.apply(s) for s in range(1 << self.n))

    def equivalent(self, other: "Circuit") -> bool:
        """Functional equivalence: both circuits compute the same function."""
        return self.n == other.n and self.permutation() == other.permutation()

    # -- display ------------------------------------------------------------
    def _bits(self, state: int) -> str:
        return " ".join(str((state >> i) & 1) for i in range(self.n))

    def truth_table(self) -> str:
        head = " ".join(f"q{i}" for i in range(self.n))
        rows = [f"{head} | {head}"]
        for s, t in enumerate(self.permutation()):
            pad = "  ".join(self._bits(s).split())
            out = "  ".join(self._bits(t).split())
            rows.append(f" {pad} |  {out}")
        return "\n".join(rows)

    def draw(self) -> str:
        """Text drawing: circled plus = target, filled dot = positive control, open dot = negative control."""
        width = len(f"q{self.n - 1}")
        rows = []
        for r in range(self.n):
            cells = []
            for g in self.gates:
                lo, hi = min(g.lines), max(g.lines)
                if r == g.target:
                    ch = "\u2295"  # target
                elif r in g.pos:
                    ch = "\u25cf"  # filled dot
                elif r in g.neg:
                    ch = "\u25cb"  # open dot
                elif lo < r < hi:
                    ch = "\u253c"  # wire crossing a gate
                else:
                    ch = "\u2500"
                cells.append("\u2500" + ch + "\u2500")
            rows.append(f"q{r}:".ljust(width + 2) + "\u2500" + "".join(cells) + "\u2500")
        return "\n".join(rows)


# ---------------------------------------------------------------------------
# Helpers for experiments
# ---------------------------------------------------------------------------
def pn_choices(lines: Sequence[int]) -> Iterator[Tuple[frozenset, frozenset]]:
    """All ways to make each given line absent, a positive or a negative control."""
    for states in product((0, 1, 2), repeat=len(lines)):
        pos = frozenset(l for l, s in zip(lines, states) if s == 1)
        neg = frozenset(l for l, s in zip(lines, states) if s == 2)
        yield pos, neg


def all_gates(n: int) -> List[Gate]:
    """Every MPMCT gate on n lines: n * 3^(n-1) gates."""
    gates = []
    for target in range(n):
        others = [l for l in range(n) if l != target]
        for pos, neg in pn_choices(others):
            gates.append(Gate(pos, neg, target))
    return gates


def random_gate(n: int, rng: Optional[random.Random] = None) -> Gate:
    rng = rng or random
    target = rng.randrange(n)
    pos, neg = set(), set()
    for l in range(n):
        if l == target:
            continue
        s = rng.randrange(3)
        if s == 1:
            pos.add(l)
        elif s == 2:
            neg.add(l)
    return Gate(frozenset(pos), frozenset(neg), target)


def random_circuit(n: int, length: int, rng: Optional[random.Random] = None) -> Circuit:
    return Circuit(n, [random_gate(n, rng) for _ in range(length)])

## Step 2: Transformation rules

In [ ]:
%%writefile rules.py
"""Transformation rules for reversible circuits (Phase 1).

Rules 1 to 5 are the complete rule set RC of Feng and Li (IEEE TCAD, 2026).
Rules 6 to 10 are the rules the paper derives from RC.

Every rewrite function takes a circuit and a position i and returns a NEW
circuit with the rule applied at position i, or None when the rule does not
match there. The input circuit is never modified.

For rules stated on gates rather than positions, a `ruleK_gates` or
`ruleK_sides` helper returns the gate lists on each side of the rule.
"""
from __future__ import annotations

from itertools import combinations
from typing import Iterable, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, X


# ===========================================================================
# Rule 1:  A A  ==  (empty)
# ===========================================================================
def rule1_cancel(c: Circuit, i: int) -> Optional[Circuit]:
    """Delete two adjacent identical gates at positions i and i+1."""
    if 0 <= i and i + 1 < len(c) and c[i] == c[i + 1]:
        return c.replace(i, i + 2, [])
    return None


def rule1_insert(c: Circuit, i: int, gate: Gate) -> Circuit:
    """Rule 1 read right to left: insert the pair `gate gate` before position i."""
    return c.replace(i, i, [gate, gate])


# ===========================================================================
# Rule 2:  G[P, N+{p}, q] G[P+{p}, N, q]  ==  G[P, N, q]
# ===========================================================================
def rule2_gates(gate: Gate, p: int) -> List[Gate]:
    """The pair A0, A1 that Rule 2 merges into `gate`, using extra line p."""
    if p in gate.lines:
        raise ValueError("p must be a line the gate does not use")
    return [G(gate.pos, gate.neg | {p}, gate.target), G(gate.pos | {p}, gate.neg, gate.target)]


def rule2_merge(c: Circuit, i: int) -> Optional[Circuit]:
    """Merge gates i and i+1 when they are A0 A1 in the order of the paper."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    extra = a1.pos - a0.pos
    if a0.target != a1.target or len(extra) != 1:
        return None
    (p,) = extra
    merged = G(a1.pos - {p}, a1.neg, a1.target)
    if rule2_gates(merged, p) == [a0, a1]:
        return c.replace(i, i + 2, [merged])
    return None


def rule2_split(c: Circuit, i: int, p: int) -> Optional[Circuit]:
    """Rule 2 read right to left: split gate i into A0 A1 on unused line p."""
    if not (0 <= i < len(c)) or not (0 <= p < c.n) or p in c[i].lines:
        return None
    return c.replace(i, i + 1, rule2_gates(c[i], p))


# ===========================================================================
# Rule 3:  A B == B A  when a control has opposite polarity in A and B
# ===========================================================================
def rule3_condition(a: Gate, b: Gate) -> bool:
    return bool(a.pos & b.neg) or bool(b.pos & a.neg)


def rule3_commute(c: Circuit, i: int) -> Optional[Circuit]:
    """Swap gates i and i+1 when they share a control of opposite polarity."""
    if 0 <= i and i + 1 < len(c) and rule3_condition(c[i], c[i + 1]):
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# ===========================================================================
# Rule 4:  A B A C1 == C2 A B A,  A = CNOT[p,q], B = CNOT[q,p]
#          (moving a swap of p and q through a gate)
# ===========================================================================
def rule4_sides(p: int, q: int, P: Iterable[int], N: Iterable[int],
                negative: bool = False) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 4.

    negative=False is condition (a): C1 has q as positive control, target p.
    negative=True  is condition (b): C1 has q as negative control, target p.
    """
    P, N = frozenset(P), frozenset(N)
    a, b = CNOT(p, q), CNOT(q, p)
    if negative:
        c1, c2 = G(P, N | {q}, p), G(P, N | {p}, q)
    else:
        c1, c2 = G(P | {q}, N, p), G(P | {p}, N, q)
    return [a, b, a, c1], [c2, a, b, a]


def _is_cnot(g: Gate) -> bool:
    return len(g.pos) == 1 and not g.neg


def rule4_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Apply Rule 4 to the four gates starting at i, in either direction."""
    if i < 0 or i + 4 > len(c):
        return None
    seg = c.gates[i:i + 4]
    # Left to right:  A B A C1  ->  C2 A B A
    if _is_cnot(seg[0]):
        (p,) = seg[0].pos
        q = seg[0].target
        c1 = seg[3]
        if c1.target == p and q in c1.controls:
            lhs, rhs = rule4_sides(p, q, c1.pos - {q}, c1.neg - {q}, q in c1.neg)
            if seg == lhs:
                return c.replace(i, i + 4, rhs)
    # Right to left:  C2 A B A  ->  A B A C1
    if _is_cnot(seg[1]):
        (p,) = seg[1].pos
        q = seg[1].target
        c2 = seg[0]
        if c2.target == q and p in c2.controls:
            lhs, rhs = rule4_sides(p, q, c2.pos - {p}, c2.neg - {p}, p in c2.neg)
            if seg == rhs:
                return c.replace(i, i + 4, lhs)
    return None


# ===========================================================================
# Rule 5:  A0 A1 B1..Bm..B1 A1 A0  ==  B'1..B'm..B'1
#          (changes the polarity of control q0)
# ===========================================================================
def rule5_sides(P: Iterable[int], N: Iterable[int], Q: Sequence[int],
                q0: int) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 5 for control sets P, N, ordered list Q and line q0."""
    P, N, Q = frozenset(P), frozenset(N), list(Q)
    Qs = frozenset(Q)
    if not Q or len(Qs) != len(Q):
        raise ValueError("Q must be a non-empty list of distinct lines")
    a0 = G(P, N | Qs, q0)
    a1 = G(P | Qs, N, q0)
    b = [G(P | {q0} | frozenset(Q[k + 1:]), N | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    bp = [G(P | frozenset(Q[k + 1:]), N | {q0} | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    lhs = [a0, a1] + b + b[-2::-1] + [a1, a0]
    rhs = bp + bp[-2::-1]
    return lhs, rhs


def rule5_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Left to right: replace the 2m+3 gate pattern starting at i by 2m-1 gates."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    Qs = a1.pos - a0.pos
    if a0.target != a1.target or not Qs or a0.neg - a1.neg != Qs:
        return None
    if a0.pos - a1.pos or a1.neg - a0.neg:
        return None
    m = len(Qs)
    end = i + 2 * m + 3
    if end > len(c):
        return None
    order = [g.target for g in c.gates[i + 2:i + 2 + m]]
    if frozenset(order) != Qs:
        return None
    lhs, rhs = rule5_sides(a0.pos, a1.neg, order, a0.target)
    if c.gates[i:end] == lhs:
        return c.replace(i, end, rhs)
    return None


def rule5_reverse(c: Circuit, i: int, Q: Sequence[int], q0: int) -> Optional[Circuit]:
    """Right to left: expand B'1..B'm..B'1 at i, given the list Q and line q0."""
    Q = list(Q)
    end = i + 2 * len(Q) - 1
    if i < 0 or not Q or end > len(c):
        return None
    b1 = c[i]
    if b1.target != Q[0] or q0 not in b1.neg:
        return None
    try:
        lhs, rhs = rule5_sides(b1.pos - frozenset(Q[1:]), b1.neg - {q0}, Q, q0)
    except ValueError:
        return None
    if c.gates[i:end] == rhs:
        return c.replace(i, end, lhs)
    return None


# ===========================================================================
# Derived rules 6 to 10
# ===========================================================================
# Rule 6:  G[P,N,q] == product over all subsets S of Q of G[P+S, N+(Q-S), q]
def rule6_gates(gate: Gate, Q: Iterable[int]) -> List[Gate]:
    Q = sorted(set(Q))
    if set(Q) & gate.lines:
        raise ValueError("Q must not overlap the lines of the gate")
    out = []
    for r in range(len(Q) + 1):
        for sub in combinations(Q, r):
            out.append(G(gate.pos | set(sub), gate.neg | (set(Q) - set(sub)), gate.target))
    return out


def rule6_expand(c: Circuit, i: int, Q: Iterable[int]) -> Optional[Circuit]:
    """Expand gate i over all polarity patterns of the unused lines Q."""
    Q = set(Q)
    if not (0 <= i < len(c)) or not Q or Q & c[i].lines or max(Q) >= c.n or min(Q) < 0:
        return None
    return c.replace(i, i + 1, rule6_gates(c[i], Q))


# Rule 7:  two gates with the same target commute
def rule7_commute(c: Circuit, i: int) -> Optional[Circuit]:
    if 0 <= i and i + 1 < len(c) and c[i].target == c[i + 1].target:
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# Rule 8:  G[P,N,q] == X[N] G[P+N, {}, q] X[N]
def rule8_gates(gate: Gate) -> List[Gate]:
    xs = [X(l) for l in sorted(gate.neg)]
    return xs + [G(gate.pos | gate.neg, (), gate.target)] + xs


def rule8_expand(c: Circuit, i: int) -> Optional[Circuit]:
    """Replace the negative controls of gate i by X gates before and after."""
    if 0 <= i < len(c) and c[i].neg:
        return c.replace(i, i + 1, rule8_gates(c[i]))
    return None


def rule8_collapse(c: Circuit, i: int) -> Optional[Circuit]:
    """Right to left: absorb X..X B X..X starting at i into one gate."""
    k = 0
    while i + k < len(c) and not c[i + k].controls:
        k += 1
    if i < 0 or k == 0 or i + 2 * k + 1 > len(c):
        return None
    b = c[i + k]
    lines = frozenset(g.target for g in c.gates[i:i + k])
    if len(lines) != k or b.neg or not lines <= b.pos:
        return None
    merged = G(b.pos - lines, lines, b.target)
    if c.gates[i:i + 2 * k + 1] == rule8_gates(merged):
        return c.replace(i, i + 2 * k + 1, [merged])
    return None


# Rule 9:  A B1 == B2 A, where A targets p and B1, B2 differ in the polarity of p
def _rule9_pair(a: Gate, b: Gate) -> Optional[Gate]:
    """If A can pass B by Rule 9, return B with the polarity of A's target flipped."""
    p = a.target
    if p not in b.controls:
        return None
    if not (a.pos <= b.pos - {p} and a.neg <= b.neg - {p}):
        return None
    if p in b.pos:
        return G(b.pos - {p}, b.neg | {p}, b.target)
    return G(b.pos | {p}, b.neg - {p}, b.target)


def rule9_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Move a gate past its neighbour, flipping one control polarity in the neighbour."""
    if i < 0 or i + 1 >= len(c):
        return None
    g1, g2 = c[i], c[i + 1]
    flipped = _rule9_pair(g1, g2)          # A B1 -> B2 A
    if flipped is not None:
        return c.replace(i, i + 2, [flipped, g1])
    flipped = _rule9_pair(g2, g1)          # B2 A -> A B1
    if flipped is not None:
        return c.replace(i, i + 2, [g2, flipped])
    return None


# Rule 10:  A B A == B A B,  A = G[P+{p}, {}, q],  B = G[P+{q}, {}, p]
def rule10_sides(P: Iterable[int], p: int, q: int) -> Tuple[List[Gate], List[Gate]]:
    P = frozenset(P)
    a, b = G(P | {p}, (), q), G(P | {q}, (), p)
    return [a, b, a], [b, a, b]


def rule10_apply(c: Circuit, i: int) -> Optional[Circuit]:
    if i < 0 or i + 3 > len(c):
        return None
    a, b = c[i], c[i + 1]
    p, q = b.target, a.target
    if a.neg or b.neg or p == q or p not in a.pos or q not in b.pos:
        return None
    if a.pos - {p} != b.pos - {q}:
        return None
    lhs, rhs = rule10_sides(a.pos - {p}, p, q)
    if c.gates[i:i + 3] == lhs:
        return c.replace(i, i + 3, rhs)
    return None

## Step 3: Verification script

In [ ]:
%%writefile verify_phase1.py
"""Phase 1 verification: every rule is checked by exhaustive simulation.

For each rule, every instance on 2 to MAX_N lines is generated. Two checks run:

  1. Soundness: the two sides of the rule compute the same permutation.
  2. Rewrite function: the instance is placed between random gates, the
     rewrite function is applied at that position, and the result must be
     the expected gate list and still compute the same function.

Run:  python verify_phase1.py          (lines 2 to 5)
      python verify_phase1.py 4        (lines 2 to 4, faster)
"""
from __future__ import annotations

import random
import sys
import time
from itertools import combinations, permutations

from revcircuit import CNOT, Circuit, G, X, all_gates, pn_choices, random_gate
import rules as R

# The circuit drawings use Unicode symbols; make sure they print on Windows too.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

MAX_N = int(sys.argv[1]) if len(sys.argv) > 1 else 5
PAIR_MAX_N = min(MAX_N, 4)          # rules over all gate pairs grow fastest
rng = random.Random(2026)


def subsets(s):
    s = sorted(s)
    for r in range(len(s) + 1):
        for sub in combinations(s, r):
            yield frozenset(sub)


# ---------------------------------------------------------------------------
# Instance generators: each yields (lhs gates, rhs gates, rewrite function)
# ---------------------------------------------------------------------------
def gen_rule1(n):
    for g in all_gates(n):
        yield [g, g], [], R.rule1_cancel


def gen_rule2(n):
    for g in all_gates(n):
        for p in range(n):
            if p not in g.lines:
                pair = R.rule2_gates(g, p)
                yield pair, [g], R.rule2_merge
                yield [g], pair, (lambda c, i, p=p: R.rule2_split(c, i, p))


def gen_rule3(n):
    gates = all_gates(n)
    for a in gates:
        for b in gates:
            if a != b and R.rule3_condition(a, b):
                yield [a, b], [b, a], R.rule3_commute


def gen_rule4(n):
    for p in range(n):
        for q in range(n):
            if p == q:
                continue
            others = [l for l in range(n) if l not in (p, q)]
            for P, N in pn_choices(others):
                for negative in (False, True):
                    lhs, rhs = R.rule4_sides(p, q, P, N, negative)
                    yield lhs, rhs, R.rule4_apply
                    yield rhs, lhs, R.rule4_apply


def gen_rule5(n):
    for q0 in range(n):
        rest = [l for l in range(n) if l != q0]
        for m in range(1, len(rest) + 1):
            for Q in permutations(rest, m):
                others = [l for l in rest if l not in Q]
                for P, N in pn_choices(others):
                    lhs, rhs = R.rule5_sides(P, N, Q, q0)
                    yield lhs, rhs, R.rule5_apply
                    yield rhs, lhs, (lambda c, i, Q=Q, q0=q0: R.rule5_reverse(c, i, Q, q0))


def gen_rule6(n):
    for g in all_gates(n):
        free = [l for l in range(n) if l not in g.lines]
        for Q in subsets(free):
            if Q:
                yield [g], R.rule6_gates(g, Q), (lambda c, i, Q=Q: R.rule6_expand(c, i, Q))


def gen_rule7(n):
    gates = all_gates(n)
    for a in gates:
        for b in gates:
            if a != b and a.target == b.target:
                yield [a, b], [b, a], R.rule7_commute


def gen_rule8(n):
    for g in all_gates(n):
        if g.neg:
            expanded = R.rule8_gates(g)
            yield [g], expanded, R.rule8_expand
            yield expanded, [g], R.rule8_collapse


def gen_rule9(n):
    for p in range(n):
        for q in range(n):
            if p == q:
                continue
            others = [l for l in range(n) if l not in (p, q)]
            for P2, N2 in pn_choices(others):
                b1 = G(P2 | {p}, N2, q)
                b2 = G(P2, N2 | {p}, q)
                for P1 in subsets(P2):
                    for N1 in subsets(N2):
                        a = G(P1, N1, p)
                        yield [a, b1], [b2, a], R.rule9_apply
                        yield [b2, a], [a, b1], R.rule9_apply
                        yield [a, b2], [b1, a], R.rule9_apply


def gen_rule10(n):
    for p in range(n):
        for q in range(n):
            if p == q:
                continue
            others = [l for l in range(n) if l not in (p, q)]
            for P in subsets(others):
                lhs, rhs = R.rule10_sides(P, p, q)
                yield lhs, rhs, R.rule10_apply


RULES = [
    ("1", "A A = empty", gen_rule1, MAX_N),
    ("2", "merge / split on one control", gen_rule2, MAX_N),
    ("3", "commute, opposite-polarity control", gen_rule3, PAIR_MAX_N),
    ("4", "move swap through a gate", gen_rule4, MAX_N),
    ("5", "change control polarity", gen_rule5, MAX_N),
    ("6", "expand over unused lines", gen_rule6, MAX_N),
    ("7", "commute, same target", gen_rule7, PAIR_MAX_N),
    ("8", "negative controls <-> X gates", gen_rule8, MAX_N),
    ("9", "move gate past neighbour", gen_rule9, MAX_N),
    ("10", "A B A = B A B", gen_rule10, MAX_N),
]


# ---------------------------------------------------------------------------
# Checks
# ---------------------------------------------------------------------------
def check_rule(gen, max_n):
    """Return (instances checked, embedded rewrites checked, first failure or None)."""
    sound = embedded = 0
    for n in range(2, max_n + 1):
        for lhs, rhs, fn in gen(n):
            left, right = Circuit(n, lhs), Circuit(n, rhs)
            if not left.equivalent(right):
                return sound, embedded, f"NOT EQUIVALENT on {n} lines: {left}  vs  {right}"
            sound += 1
            # Embedded rewrite test (all instances on small n, a sample on larger n).
            if n <= 3 or rng.random() < 0.1:
                prefix = [random_gate(n, rng) for _ in range(rng.randrange(4))]
                suffix = [random_gate(n, rng) for _ in range(rng.randrange(4))]
                before = Circuit(n, prefix + lhs + suffix)
                after = fn(before, len(prefix))
                if after is None:
                    return sound, embedded, f"rewrite did not match: {left}"
                if after.gates != prefix + rhs + suffix:
                    return sound, embedded, f"rewrite gave unexpected gates: {left}"
                if not after.equivalent(before):
                    return sound, embedded, f"rewrite changed the function: {left}"
                embedded += 1
    return sound, embedded, None


def main() -> int:
    print(f"Phase 1 verification, lines 2 to {MAX_N} "
          f"(pair rules 3 and 7: 2 to {PAIR_MAX_N})\n")
    print(f"{'Rule':<5}{'Statement':<38}{'Instances':>10}{'Rewrites':>10}  Result")
    print("-" * 72)
    failures = 0
    start = time.time()
    for name, text, gen, max_n in RULES:
        sound, embedded, err = check_rule(gen, max_n)
        result = "PASS" if err is None else "FAIL"
        failures += err is not None
        print(f"{name:<5}{text:<38}{sound:>10}{embedded:>10}  {result}")
        if err:
            print(f"      {err}")
    print("-" * 72)
    print(f"Time: {time.time() - start:.1f} s\n")

    # Sanity check: the equivalence test must reject a wrong rewrite.
    wrong_a = Circuit(2, [CNOT(0, 1), CNOT(1, 0)])
    wrong_b = Circuit(2, [CNOT(1, 0), CNOT(0, 1)])
    detected = not wrong_a.equivalent(wrong_b)
    failures += not detected
    print("Sanity check, non-commuting gates detected as different:",
          "PASS" if detected else "FAIL")

    # Worked example: Example 2 of Feng and Li.
    print("\nWorked example (Example 2 of the paper):")
    print("X[q0] CNOT[q0,q1] X[q0]  ->  negatively controlled NOT\n")
    c = Circuit(2, [X(0), CNOT(0, 1), X(0)])
    steps = [("Start", c)]
    c = R.rule2_split(c, 0, 1); steps.append(("Rule 2 on the first X", c))
    c = R.rule2_split(c, 3, 1); steps.append(("Rule 2 on the second X", c))
    c = R.rule3_commute(c, 3);  steps.append(("Rule 3 on the last two gates", c))
    c = R.rule5_apply(c, 0);    steps.append(("Rule 5 on the whole circuit", c))
    for label, circ in steps:
        print(f"{label}:  {circ}")
        print(circ.draw())
        print()
    ok = (c == Circuit(2, [G((), {0}, 1)])) and all(s.equivalent(steps[0][1]) for _, s in steps)
    failures += not ok
    print("Example 2 reproduced:", "PASS" if ok else "FAIL")

    print("\nAll checks passed." if failures == 0 else f"\n{failures} check(s) FAILED.")
    return 1 if failures else 0


if __name__ == "__main__":
    sys.exit(main())

## Step 4: Run the verification

Every rule should show **PASS**, and the last line should read `All checks passed.`

In [ ]:
!python verify_phase1.py

## Step 5: Try it yourself

This cell builds a small circuit, draws it, and applies rules to it by hand.
Lines are numbered from 0, so `CNOT(0, 1)` has control `q0` and target `q1`.

Drawing symbols: `⊕` target, `●` positive control, `○` negative control.

In [ ]:
import importlib
import revcircuit, rules
importlib.reload(revcircuit); importlib.reload(rules)

from revcircuit import Circuit, G, X, CNOT, TOF
import rules as R

# A 3-line circuit: an X gate, a Toffoli gate, then two identical CNOT gates.
c = Circuit(3, [X(0), TOF(0, 1, 2), CNOT(0, 1), CNOT(0, 1)])
print("Original circuit:", c)
print(c.draw())
print()

# Rule 1 removes the two identical CNOT gates at positions 2 and 3.
c2 = R.rule1_cancel(c, 2)
print("After Rule 1:", c2)
print(c2.draw())
print()

# Rule 9 moves the X gate past the Toffoli gate; the control on q0 becomes negative.
c3 = R.rule9_apply(c2, 0)
print("After Rule 9:", c3)
print(c3.draw())
print()

print("Same function as the original?", c.equivalent(c2) and c.equivalent(c3))
print()
print("Truth table of the original circuit (inputs | outputs):")
print(c.truth_table())